[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaronJenish/zawdwadw/blob/main/Experiment_29_VAE_GAN_Sample_Generation.ipynb)


# Experiment 29: VAE/GAN Sample Generation

CSA47 – Deep Learning. Run the setup cell first, then execute the experiment cells in order.


In [ ]:
# Common setup – run this cell first
import os, time, warnings, math
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, mean_absolute_error, mean_squared_error)
np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 4)
print("Setup complete")

## Experiment 29 – VAE/GAN Sample Generation

In [ ]:
# Fashion-MNIST VAE and DCGAN; subset training for Colab runtime
import tensorflow as tf, time
(xtrain,_),(_,_) = tf.keras.datasets.fashion_mnist.load_data()
xtrain=xtrain[:12000].astype("float32")/255.
xtrain=xtrain[...,None]
# VAE
latent_dim=2
inp=tf.keras.Input((28,28,1)); z=tf.keras.layers.Flatten()(inp); z=tf.keras.layers.Dense(128,activation="relu")(z)
mu=tf.keras.layers.Dense(latent_dim)(z); logvar=tf.keras.layers.Dense(latent_dim)(z)
def sample_latent(args):
    mu,lv=args; eps=tf.random.normal(tf.shape(mu)); return mu+tf.exp(.5*lv)*eps
zsample=tf.keras.layers.Lambda(sample_latent)([mu,logvar])
enc=tf.keras.Model(inp,[mu,logvar,zsample])
zin=tf.keras.Input((latent_dim,)); d=tf.keras.layers.Dense(128,activation="relu")(zin); d=tf.keras.layers.Dense(784,activation="sigmoid")(d); out=tf.keras.layers.Reshape((28,28,1))(d)
dec=tf.keras.Model(zin,out)
class VAE(tf.keras.Model):
    def __init__(self): super().__init__(); self.enc=enc; self.dec=dec
    def train_step(self,data):
        with tf.GradientTape() as tape:
            m,l,z=self.enc(data,training=True); rec=self.dec(z,training=True)
            recon=tf.reduce_mean(tf.reduce_sum(tf.keras.losses.binary_crossentropy(data,rec),axis=(1,2)))
            kl=-.5*tf.reduce_mean(tf.reduce_sum(1+l-tf.square(m)-tf.exp(l),axis=1)); loss=recon+kl
        grads=tape.gradient(loss,self.trainable_variables); self.optimizer.apply_gradients(zip(grads,self.trainable_variables))
        return {"loss":loss,"reconstruction":recon,"kl":kl}
vae=VAE(); vae.compile(optimizer="adam"); vh=vae.fit(xtrain,batch_size=128,epochs=3,verbose=1)
# Generate VAE samples
samples=dec.predict(np.random.normal(size=(10,2)),verbose=0)
fig,axs=plt.subplots(1,10,figsize=(12,2))
for ax,img in zip(axs,samples): ax.imshow(img.squeeze(),cmap="gray"); ax.axis("off")
plt.show()
# Compact DCGAN
gen=tf.keras.Sequential([tf.keras.layers.Input((100,)),tf.keras.layers.Dense(7*7*128),tf.keras.layers.Reshape((7,7,128)),
 tf.keras.layers.Conv2DTranspose(64,4,strides=2,padding="same"),tf.keras.layers.LeakyReLU(.2),
 tf.keras.layers.Conv2DTranspose(1,4,strides=2,padding="same",activation="sigmoid")])
disc=tf.keras.Sequential([tf.keras.layers.Input((28,28,1)),tf.keras.layers.Conv2D(64,4,strides=2,padding="same"),tf.keras.layers.LeakyReLU(.2),
 tf.keras.layers.Flatten(),tf.keras.layers.Dense(1)])
optg=tf.keras.optimizers.Adam(1e-4); optd=tf.keras.optimizers.Adam(1e-4); bce=tf.keras.losses.BinaryCrossentropy(from_logits=True)
g_losses=[]; d_losses=[]
for epoch in range(3):
    for real in tf.data.Dataset.from_tensor_slices(xtrain).shuffle(5000).batch(128):
        noise=tf.random.normal((len(real),100))
        with tf.GradientTape() as gt, tf.GradientTape() as dt:
            fake=gen(noise,training=True); realout=disc(real,training=True); fakeout=disc(fake,training=True)
            gl=bce(tf.ones_like(fakeout),fakeout); dl=bce(tf.ones_like(realout),realout)+bce(tf.zeros_like(fakeout),fakeout)
        optg.apply_gradients(zip(gt.gradient(gl,gen.trainable_variables),gen.trainable_variables))
        optd.apply_gradients(zip(dt.gradient(dl,disc.trainable_variables),disc.trainable_variables))
        g_losses.append(float(gl)); d_losses.append(float(dl))
    print("Epoch",epoch+1,"G loss",np.mean(g_losses[-len(xtrain)//128:]),"D loss",np.mean(d_losses[-len(xtrain)//128:]))
gen_samples=gen.predict(np.random.normal(size=(10,100)),verbose=0)
fig,axs=plt.subplots(1,10,figsize=(12,2))
for ax,img in zip(axs,gen_samples): ax.imshow(img.squeeze(),cmap="gray"); ax.axis("off")
plt.show()
plt.plot(g_losses,label="Generator"); plt.plot(d_losses,label="Discriminator"); plt.legend(); plt.title("GAN losses"); plt.show()